# Предсказание качества вина по его параметрам

## 1. Загрузка данных

In [1]:
import csv
import numpy as np

wine_path = '/home/avtotka/dev/ai/pytorch/winequality-white.csv'
wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=';', skiprows=1)

wineq_numpy

array([[ 7.  ,  0.27,  0.36, ...,  0.45,  8.8 ,  6.  ],
       [ 6.3 ,  0.3 ,  0.34, ...,  0.49,  9.5 ,  6.  ],
       [ 8.1 ,  0.28,  0.4 , ...,  0.44, 10.1 ,  6.  ],
       ...,
       [ 6.5 ,  0.24,  0.19, ...,  0.46,  9.4 ,  6.  ],
       [ 5.5 ,  0.29,  0.3 , ...,  0.38, 12.8 ,  7.  ],
       [ 6.  ,  0.21,  0.38, ...,  0.32, 11.8 ,  6.  ]],
      shape=(4898, 12), dtype=float32)

In [2]:
import torch

wineq = torch.from_numpy(wineq_numpy)
wineq.shape, wineq.dtype

(torch.Size([4898, 12]), torch.float32)

## 2. Нормализация и подготовка

In [3]:
source = wineq[:, :-1] # все строки, кроме последнего столбца
target = wineq[:, -1] # все строки, последний столбец

source.shape, target.shape

(torch.Size([4898, 11]), torch.Size([4898]))

In [4]:
target = target.long() - 3 # минимальная оценка 3

source_mean = torch.mean(source, dim=0) # среднее для каждого из столбцов
source_var = torch.var(source, dim=0) # стандартное отклонение для каждого из столбцов
source_normalized = (source - source_mean) / torch.sqrt(source_var)

source_normalized = source_normalized

## 3. Разбиение на обучающий и валидирующий наборы

In [5]:
n_samples = wineq.shape[0]
n_val = int(0.2 * n_samples)

shuffled_indices = torch.randperm(n_samples)

train_indices = shuffled_indices[:-n_val]
val_indices = shuffled_indices[-n_val:]

w_train = source_normalized[train_indices]
wq_train = target[train_indices]

w_val = source_normalized[val_indices]
wq_val = target[val_indices]

print(f'Train: {w_train.shape}, {wq_train.shape}')
print(f'Validate: {w_val.shape}, {wq_val.shape}')

Train: torch.Size([3919, 11]), torch.Size([3919])
Validate: torch.Size([979, 11]), torch.Size([979])


## 4. Построение модели

In [6]:
import torch.nn as nn

def training_loop_nn(n_epochs, optimizer, model, loss_fn, w_train, wq_train, w_val, wq_val):
    for epoch in range(1, n_epochs + 1):
        wq_p_train = model(w_train)
        loss_train = loss_fn(wq_p_train, wq_train)

        wq_p_val = model(w_val)
        loss_val = loss_fn(wq_p_val, wq_val)

        optimizer.zero_grad()
        loss_train.backward()
        optimizer.step()

        if epoch <= 3 or epoch % 1000 == 0:
            print(f'Epoch {epoch}, Training loss {loss_train.item()}, Validation loss {loss_val.item()}')

In [7]:
from collections import OrderedDict

seq_model = nn.Sequential(
    OrderedDict([
        ('hidden_linear', nn.Linear(11, 64)),
        ('hidden_activation', nn.Tanh()),
        ('dropout', nn.Dropout(0.3)),
        ('output_activation', nn.Linear(64, 7))
]))
seq_model

Sequential(
  (hidden_linear): Linear(in_features=11, out_features=64, bias=True)
  (hidden_activation): Tanh()
  (dropout): Dropout(p=0.3, inplace=False)
  (output_activation): Linear(in_features=64, out_features=7, bias=True)
)

In [8]:
import torch.optim as optim

optimizer = optim.Adam(seq_model.parameters(), lr=1e-4)
training_loop_nn(
    n_epochs = 6000,
    optimizer = optimizer,
    model = seq_model,
    loss_fn = nn.CrossEntropyLoss(),
    w_train = w_train,
    wq_train = wq_train,
    w_val = w_val,
    wq_val = wq_val)

Epoch 1, Training loss 2.027252197265625, Validation loss 2.0255539417266846
Epoch 2, Training loss 2.027255058288574, Validation loss 2.023648977279663
Epoch 3, Training loss 2.025756359100342, Validation loss 2.022179126739502
Epoch 1000, Training loss 1.2172187566757202, Validation loss 1.217488169670105
Epoch 2000, Training loss 1.113550066947937, Validation loss 1.1124478578567505
Epoch 3000, Training loss 1.089256763458252, Validation loss 1.0896430015563965
Epoch 4000, Training loss 1.0820916891098022, Validation loss 1.0836596488952637
Epoch 5000, Training loss 1.0661290884017944, Validation loss 1.0763975381851196
Epoch 6000, Training loss 1.0613588094711304, Validation loss 1.0664851665496826
